# Titanic V2: XGBoost Avançado & Engenharia de Dados
Neste notebook o fluxo está rodando de ponta a ponta sem interrupções. As bases de treino e teste foram unificadas matematicamente para não haver perda de dados escalares e utilizamos as estratégias recomendadas por Grandmasters: KNN para prever Idades, Frequência de Ticket, e extração da rede topológica das Cabines.

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# 1. Unificando os Conjuntos
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
y_train = train['Survived']
passenger_id_test = test['PassengerId']

all_data = pd.concat([train.drop(columns=['Survived']), test], ignore_index=True)
print("Tamanho total unificado:", all_data.shape)

### Fase 1: Feature Engineering Avançada

In [ ]:
# A. Ticket Freqüência & Topologia do Navio (Decks)
all_data['Ticket_Freq'] = all_data.groupby('Ticket')['Ticket'].transform('count')
all_data['Deck'] = all_data['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'M')

# B. Títulos
all_data['Title'] = all_data['Name'].apply(lambda x: re.search(' ([A-Za-z]+)\.', x).group(1))
all_data['Title'] = all_data['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
all_data['Title'] = all_data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# C. Tratamento Financeiro e Binning
all_data['Fare'] = all_data['Fare'].fillna(all_data['Fare'].median())
all_data['Fare_Bin'] = pd.qcut(all_data['Fare'], 5, labels=False)

# D. Terapia Familiar (Agrupando de verdade)
all_data['FamilySize'] = all_data['SibSp'] + all_data['Parch'] + 1
def categorize_family(size):
    if size == 1: return 'Alone'
    elif size <= 4: return 'Small'
    else: return 'Large'
    
all_data['FamilyGroup'] = all_data['FamilySize'].apply(categorize_family)

### Fase 2: Imputação Multivariada (KNN Imputer)

In [ ]:
from sklearn.impute import KNNImputer

# Numéricos amados pela estatística de vizinhança 
all_data['Sex_Bin'] = all_data['Sex'].map({'male': 0, 'female': 1})
all_data['Title_Num'] = all_data['Title'].astype('category').cat.codes

knn_cols = ['Pclass', 'Sex_Bin', 'SibSp', 'Parch', 'Fare', 'Title_Num', 'Age']
df_knn = all_data[knn_cols].copy()

# Usando as vidas de 5 pessoas similares para adivinhar a Idade da que falta
imputer = KNNImputer(n_neighbors=5)
all_data['Age'] = pd.DataFrame(imputer.fit_transform(df_knn), columns=df_knn.columns)['Age']

# Idade Binning (discretizando assim como a tarifa)
all_data['Age_Bin'] = pd.qcut(all_data['Age'], 5, labels=False)
print("Idades faltantes restantes:", all_data['Age'].isnull().sum())

### Fase 3: Limpeza Final e Encoding das Dummies

In [ ]:
# Preencher Embarked 
all_data['Embarked'] = all_data['Embarked'].fillna(all_data['Embarked'].mode()[0])

# Jogar o lixo fora para as árvores respirarem!
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch', 
             'FamilySize', 'Fare', 'Age', 'Title_Num', 'Sex_Bin']
all_data = all_data.drop(columns=drop_cols)

# Converter todas as variáveis categóricas em Variáveis Dummy (Verdadeiro ou Falso booleano)
all_data = pd.get_dummies(all_data, drop_first=True)

# SEPARANDO O NAVIO! Recuperando a fórmula Original
X_train = all_data.iloc[:len(train)]
X_test = all_data.iloc[len(train):]

### Fase 4: O Monstro (XGBoost) e Submissão Final

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

# 1. Instanciando o Extreme Gradient Boosting
# Coloquei uma taxa de aprendizado devagar (0.05) para as árvores irem subindo minuciosamente
xgb_model = XGBClassifier(
    n_estimators=150, 
    max_depth=4, 
    learning_rate=0.05, 
    random_state=42, 
    eval_metric='logloss'
)

# CROSS-VALIDATION STATS: O quão bom o modelo real é? (Treina em fatias giratórias do próprio treino)
scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"XGBoost Validation Accuracy Média Máxima: {scores.mean()*100:.2f}% (Std: {scores.std():.3f})")

# 2. Treinando 100% da força com o total dos dados
xgb_model.fit(X_train, y_train)

# 3. Submetendo para Submission V2
preds = xgb_model.predict(X_test)
sub = pd.DataFrame({'PassengerId': passenger_id_test, 'Survived': preds})
sub.to_csv('submission2_advanced.csv', index=False)

print("Arquivo submission2_advanced.csv CRIADO com absoluto sucesso!")

### Explicabilidade: O que a Inteligência Artificial aprendeu?
Este gráfico nos mostra por dentro da 'Caixa Preta' quais foram as variáveis que o nosso XGBoost considerou mais vitais para sentenciar vida ou morte.

In [ ]:
import matplotlib.pyplot as plt
from xgboost import plot_importance
# Gráfico de Importância das Features do modelo XGBoost
plt.rcParams['figure.figsize'] = [10, 6]
plot_importance(xgb_model, max_num_features=10, title='Top 10 Features mais importantes (XGBoost)', show_values=False, importance_type='weight')
plt.show()